# ML Pipelines, Serving & Monitoring

- [Data Pipelines and Scaling](#Data-Pipelines-and-Scaling)
  - [ETL pipeline (pandas): extract, validate, transform, load](#ETL-pipeline-(pandas):-extract,-validate,-transform,-load)
  - [Setup](#Setup)
  - [Config](#Config)
  - [Helper functions](#Helper-functions)
  - [Data generation](#Data-generation)
  - [Extraction (I)](#Extraction-(I))
  - [Validation](#Validation)
  - [Transform (I)](#Transform-(I))
  - [Visual analysis](#Visual-analysis)
  - [Load (I)](#Load-(I))
  - [Extract (II) and EDA checks](#Extract-(II)-and-EDA-checks)
  - [Load (II)](#Load-(II))
  - [Transform (II)](#Transform-(II))
  - [Scaling to PySpark: lazy evaluation, shuffles, partitioning](#Scaling-to-PySpark:-lazy-evaluation,-shuffles,-partitioning)
  - [Feature engineering at scale with PySpark](#Feature-engineering-at-scale-with-PySpark)
- [Model Serving and Deployment](#Model-Serving-and-Deployment)
  - [Batch vs. real-time inference](#Batch-vs.-real-time-inference)
  - [Serialization: saving a trained model as a portable artifact](#Serialization:-saving-a-trained-model-as-a-portable-artifact)
  - [Wrapping a model as a real-time API (FastAPI)](#Wrapping-a-model-as-a-real-time-API-(FastAPI))
  - [Containerizing with Docker](#Containerizing-with-Docker)
  - [Model registry and versioning](#Model-registry-and-versioning)
  - [Scaling and monitoring a live model](#Scaling-and-monitoring-a-live-model)
  - [Batch vs. real-time, summarized](#Batch-vs.-real-time,-summarized)
- [Drift Monitoring and Retraining](#Drift-Monitoring-and-Retraining)
  - [Types of drift](#Types-of-drift)
  - [PSI (Population Stability Index), worked by hand](#PSI-(Population-Stability-Index),-worked-by-hand)
  - [KS-test (Kolmogorov-Smirnov), worked by hand](#KS-test-(Kolmogorov-Smirnov),-worked-by-hand)
  - [What to actually monitor in production](#What-to-actually-monitor-in-production)
  - [Retraining triggers](#Retraining-triggers)
  - [Automated retraining triggers](#Automated-retraining-triggers)
- [MLOps Tooling and Automation](#MLOps-Tooling-and-Automation)
  - [Experiment tracking and model registry (MLflow)](#Experiment-tracking-and-model-registry-(MLflow))
  - [CI/CD for ML (GitHub Actions)](#CI/CD-for-ML-(GitHub-Actions))
  - [Dataiku: flow, scenarios and automation](#Dataiku:-flow,-scenarios-and-automation)
  - [Cloud basics for ML on AWS](#Cloud-basics-for-ML-on-AWS)
  - [Dashboards and BI for model monitoring (Power BI)](#Dashboards-and-BI-for-model-monitoring-(Power-BI))

### Data Pipelines and Scaling

Build a full extract/validate/transform/load pipeline single-node (pandas), then what changes when that same pipeline needs to scale past one machine (PySpark): lazy evaluation, shuffle cost, partition sizing, and how to measure whether distributing it actually helped.

#### ETL pipeline (pandas): extract, validate, transform, load

#### Setup

In [1]:
!pip install pandas==2.2.2 numpy==2.0.2 matplotlib==3.10.0 scikit-learn==1.6.1

#### Config

In [ ]:
from pathlib import Path
import logging, random
import numpy as np

# Colab working dirs
ROOT = Path("/content")
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "output"
FIG_DIR = OUT_DIR / "figs"
for p in [DATA_DIR, OUT_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Reproducibility
RNG_SEED = 42
np.random.seed(RNG_SEED)
random.seed(RNG_SEED)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

#### Helper functions

In [ ]:
import json
import pandas as pd

def _ts(s: str) -> pd.Timestamp:
    return pd.to_datetime(s, utc=False, errors="coerce")

def save_json(obj, path: Path):
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")

#### Data generation

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta

def bootstrap_demo_data():
    sales_csv = DATA_DIR / "sales.csv"
    events_json = DATA_DIR / "events.json"
    customers_db = DATA_DIR / "customers.db"

    if sales_csv.exists() and events_json.exists() and customers_db.exists():
        logging.info("Demo data already present. Skipping bootstrap.")
        return

    logging.info("Bootstrapping demo data...")
    # Customers
    n_customers = 400
    countries = ["IN", "US", "UK", "DE", "SG"]
    cities_by_country = {
        "IN": ["Delhi", "Bengaluru", "Mumbai", "Hyderabad"],
        "US": ["NYC", "SF", "Austin"],
        "UK": ["London", "Manchester"],
        "DE": ["Berlin", "Munich"],
        "SG": ["Singapore"]
    }

    start_date = datetime(2023, 9, 1)
    end_date = datetime(2024, 9, 1)
    days = (end_date - start_date).days

    cust_rows = []
    for cid in range(1001, 1001 + n_customers):
        country = random.choice(countries)
        city = random.choice(cities_by_country[country])
        signup = start_date + timedelta(days=int(np.random.uniform(0, max(1, days*0.6))))
        cust_rows.append({
            "customer_id": cid,
            "country": country,
            "city": city,
            "signup_date": signup.date().isoformat()
        })

    # SQLite customers table
    conn = sqlite3.connect(customers_db)
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS customers (
            customer_id INTEGER PRIMARY KEY,
            country TEXT,
            city TEXT,
            signup_date TEXT
        )
    """)
    cur.executemany(
        "INSERT OR REPLACE INTO customers (customer_id, country, city, signup_date) VALUES (?, ?, ?, ?)",
        [(r["customer_id"], r["country"], r["city"], r["signup_date"]) for r in cust_rows]
    )
    conn.commit()
    conn.close()

    # Sales CSV with realistic noise/outliers
    sale_rows = []
    sale_id = 1
    for r in cust_rows:
        base_lambda = 10
        signup_dt = _ts(r["signup_date"])
        active_days = max(1, (end_date - max(start_date, signup_dt.to_pydatetime())).days)
        expected = max(0.5, base_lambda * (active_days / days))
        n = np.random.poisson(lam=expected)

        for _ in range(n):
            ts = start_date + timedelta(days=int(np.random.uniform(0, days)))
            if ts.date() < signup_dt.date():
                continue
            amt = np.random.lognormal(mean=5.0, sigma=0.6)
            if np.random.rand() < 0.02:
                amt = np.nan
            if np.random.rand() < 0.01:
                amt = -abs(amt)

            sale_rows.append({
                "sale_id": sale_id,
                "customer_id": r["customer_id"],
                "amount": amt,
                "ts": ts.isoformat(timespec="seconds")
            })
            sale_id += 1

    pd.DataFrame(sale_rows).sort_values("ts").to_csv(sales_csv, index=False)

    # Events JSON
    event_types = ["login", "view", "cart_add", "wishlist"]
    ev_rows = []
    ev_id = 1
    for r in cust_rows:
        m = np.random.poisson(lam=5.0)
        for _ in range(m):
            ts = start_date + timedelta(days=int(np.random.uniform(0, days)))
            if ts.date() < _ts(r["signup_date"]).date():
                continue
            ev_rows.append({
                "event_id": ev_id,
                "customer_id": r["customer_id"],
                "event_type": random.choice(event_types),
                "ts": ts.isoformat(timespec="seconds")
            })
            ev_id += 1

    save_json(ev_rows, events_json)
    logging.info("Bootstrap complete")

bootstrap_demo_data()

#### Extraction (I)

In [ ]:
def extract_sales_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    logging.info("Extracted sales CSV: %s rows, %s cols", *df.shape)
    return df

def extract_events_json(path: Path) -> pd.DataFrame:
    data = json.loads(path.read_text(encoding="utf-8"))
    df = pd.json_normalize(data)
    logging.info("Extracted events JSON: %s rows, %s cols", *df.shape)
    return df

def extract_customers_sqlite(path: Path) -> pd.DataFrame:
    conn = sqlite3.connect(path)
    df = pd.read_sql("SELECT * FROM customers", conn)
    conn.close()
    logging.info("Extracted customers SQLite: %s rows, %s cols", *df.shape)
    return df

sales = extract_sales_csv(DATA_DIR / "sales.csv")
events = extract_events_json(DATA_DIR / "events.json")
customers = extract_customers_sqlite(DATA_DIR / "customers.db")

sales.head(3), events.head(3), customers.head(3)

#### Validation

In [ ]:
def validate_sales(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in ["sale_id", "customer_id", "amount", "ts"]:
        if c not in df.columns:
            raise ValueError(f"sales missing required column: {c}")
    df["sale_id"] = pd.to_numeric(df["sale_id"], errors="coerce").astype("Int64")
    df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce").astype("Int64")
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")

    before = len(df)
    df = df.dropna(subset=["sale_id", "customer_id", "ts"]).reset_index(drop=True)

    # Negative / zero -> NaN, cap extreme outliers
    df.loc[df["amount"] <= 0, "amount"] = np.nan
    cap = df["amount"].quantile(0.95) * 5 if df["amount"].notna().sum() > 0 else np.nan
    df.loc[df["amount"] > cap, "amount"] = cap

    # Fill amounts: customer median -> global median
    df["amount"] = df.groupby("customer_id")["amount"].transform(lambda s: s.fillna(s.median()))
    df["amount"] = df["amount"].fillna(df["amount"].median())

    # Deduplicate by latest ts per sale_id
    df = df.sort_values(["sale_id", "ts"]).drop_duplicates(subset=["sale_id"], keep="last")
    logging.info("Validated sales: dropped %d bad/dup rows", before - len(df))
    return df

def validate_events(df: pd.DataFrame) -> pd.DataFrame:
    req = {"event_id", "customer_id", "event_type", "ts"}
    if not req.issubset(df.columns):
        raise ValueError(f"events missing required columns: {req - set(df.columns)}")
    df = df.copy()
    df["event_id"] = pd.to_numeric(df["event_id"], errors="coerce").astype("Int64")
    df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce").astype("Int64")
    df["event_type"] = df["event_type"].astype("string")
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")

    before = len(df)
    df = df.dropna(subset=["event_id", "customer_id", "event_type", "ts"])
    df = df.sort_values(["event_id", "ts"]).drop_duplicates(subset=["event_id"], keep="last")
    logging.info("Validated events: dropped %d bad/dup rows", before - len(df))
    return df

def validate_customers(df: pd.DataFrame) -> pd.DataFrame:
    req = {"customer_id", "country", "city", "signup_date"}
    if not req.issubset(df.columns):
        raise ValueError(f"customers missing required columns: {req - set(df.columns)}")
    df = df.copy()
    df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce").astype("Int64")
    df["country"] = df["country"].astype("string")
    df["city"] = df["city"].astype("string")
    df["signup_date"] = pd.to_datetime(df["signup_date"], errors="coerce").dt.date

    before = len(df)
    df = df.dropna(subset=["customer_id", "country", "city", "signup_date"])
    df = df.drop_duplicates(subset=["customer_id"], keep="last")
    logging.info("Validated customers: dropped %d bad/dup rows", before - len(df))
    return df

sales_v = validate_sales(sales)
events_v = validate_events(events)
customers_v = validate_customers(customers)

sales_v.head(3), events_v.head(3), customers_v.head(3)

#### Transform (I)

In [ ]:
def transform_make_features(sales: pd.DataFrame,
                            customers: pd.DataFrame,
                            cutoff: pd.Timestamp,
                            obs_days: int,
                            label_days: int):
    sales = sales.copy()
    customers = customers.copy()

    obs_start = cutoff - timedelta(days=obs_days)
    label_end = cutoff + timedelta(days=label_days)

    sales["ts"] = pd.to_datetime(sales["ts"])
    obs_mask = (sales["ts"] >= obs_start) & (sales["ts"] < cutoff)
    label_mask = (sales["ts"] >= cutoff) & (sales["ts"] < label_end)

    sales_obs = sales.loc[obs_mask].copy()
    sales_label = sales.loc[label_mask].copy()

    # Daily time series for viz
    daily_obs = (sales_obs.assign(day=sales_obs["ts"].dt.date)
                 .groupby("day", as_index=False)["amount"].sum()
                 .rename(columns={"amount": "daily_sales"}))

    # RFM-like features
    last_ts = sales_obs.groupby("customer_id")["ts"].max().to_frame("last_ts").reset_index()
    last_ts["recency_days"] = (cutoff - last_ts["last_ts"]).dt.days

    agg = sales_obs.groupby("customer_id").agg(
        freq=("sale_id", "count"),
        monetary=("amount", "sum"),
        avg_amount=("amount", "mean"),
        max_amount=("amount", "max")
    ).reset_index()

    def avg_gap(group):
        g = group.sort_values("ts")["ts"].values
        if len(g) < 2:
            return np.nan
        diffs = np.diff(g).astype("timedelta64[D]").astype(float)
        return float(np.mean(diffs))

    gaps = sales_obs.groupby("customer_id", group_keys=False).apply(avg_gap, include_groups=False).to_frame("avg_gap_days").reset_index()

    feats = customers[["customer_id", "country", "city", "signup_date"]].merge(agg, on="customer_id", how="left")
    feats = feats.merge(last_ts[["customer_id", "recency_days"]], on="customer_id", how="left")
    feats = feats.merge(gaps, on="customer_id", how="left")

    feats["freq"] = feats["freq"].fillna(0).astype(int)
    for c in ["monetary", "avg_amount", "max_amount", "recency_days", "avg_gap_days"]:
        feats[c] = feats[c].astype(float)

    feats["tenure_days"] = (pd.to_datetime(cutoff.date()) - pd.to_datetime(feats["signup_date"])).dt.days.clip(lower=0)

    has_future_purchase = sales_label.groupby("customer_id")["sale_id"].count().reindex(feats["customer_id"]).fillna(0)
    feats["label_churn"] = (has_future_purchase.values == 0).astype(int)

    feats = feats[[
        "customer_id", "country", "city", "tenure_days",
        "freq", "monetary", "avg_amount", "max_amount",
        "recency_days", "avg_gap_days", "label_churn"
    ]]

    return feats, daily_obs, sales_obs

default_cutoff = pd.to_datetime(sales_v["ts"].quantile(0.75)).normalize()
obs_days, label_days = 200, 60

feats, daily_obs, sales_obs = transform_make_features(
    sales=sales_v, customers=customers_v,
    cutoff=default_cutoff, obs_days=obs_days, label_days=label_days
)

default_cutoff, feats.shape, sales_obs.shape, daily_obs.shape

#### Visual analysis

In [ ]:
import matplotlib.pyplot as plt

def plot_missingness(df: pd.DataFrame, title: str, out_path: Path):
    miss = df.isna().mean()
    miss = miss[miss > 0].sort_values(ascending=False)  # only plot columns with missing
    if miss.empty:
        print(f"No missing values found in: {title}")
        return
    plt.figure(figsize=(8, 4))
    miss.plot(kind="bar")
    plt.title(title)
    plt.ylabel("Fraction Missing")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()

def plot_amount_hist(sales_obs: pd.DataFrame, out_path: Path):
    plt.figure(figsize=(6, 4))
    plt.hist(sales_obs["amount"].dropna(), bins=40)
    plt.title("Amount Distribution (Observation Window)")
    plt.xlabel("Amount")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()

def plot_daily_sales(daily_obs: pd.DataFrame, out_path: Path):
    plt.figure(figsize=(8, 4))
    plt.plot(pd.to_datetime(daily_obs["day"]), daily_obs["daily_sales"])
    plt.title("Daily Sales (Observation Window)")
    plt.xlabel("Day")
    plt.ylabel("Sales")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()

def plot_corr_heatmap(feats: pd.DataFrame, out_path: Path):
    numeric = feats.select_dtypes(include=[np.number])
    if numeric.shape[1] < 2:
        return
    corr = numeric.corr()
    plt.figure(figsize=(6, 5))
    plt.imshow(corr, aspect="auto")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
    plt.yticks(range(len(corr.index)), corr.index)
    plt.colorbar()
    plt.title("Feature Correlation (Numeric)")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()

# Generate and show inline
plot_missingness(sales_v, "Missingness: Sales (All)", FIG_DIR / "missing_sales.png")
plot_missingness(customers_v, "Missingness: Customers", FIG_DIR / "missing_customers.png")
plot_amount_hist(sales_obs, FIG_DIR / "amount_hist_obs.png")
plot_daily_sales(daily_obs, FIG_DIR / "daily_sales_obs.png")
plot_corr_heatmap(feats.drop(columns=["customer_id"]), FIG_DIR / "feature_corr.png")

from IPython.display import Image, display
for f in FIG_DIR.glob("*.png"):
    display(Image(f))

#### Load (I)

In [ ]:
def dump_outputs(feats: pd.DataFrame, sales_obs: pd.DataFrame, daily_obs: pd.DataFrame,
                 cutoff: pd.Timestamp, obs_days: int, label_days: int):
    feats.to_csv(OUT_DIR / "features.csv", index=False)
    feats.to_parquet(OUT_DIR / "features.parquet", index=False)
    feats.to_json(OUT_DIR / "features.json", orient="records", indent=2)

    sales_obs.to_csv(OUT_DIR / "sales_observation.csv", index=False)
    sales_obs.to_parquet(OUT_DIR / "sales_observation.parquet", index=False)
    sales_obs.to_json(OUT_DIR / "sales_observation.json", orient="records", indent=2)

    daily_obs.to_csv(OUT_DIR / "daily_sales.csv", index=False)

    data_dict = {
        "features": {
            "customer_id": "Unique customer key",
            "country": "Country code",
            "city": "City name",
            "tenure_days": "Days since signup until cutoff",
            "freq": "Purchase count in observation window",
            "monetary": "Total spend in observation window",
            "avg_amount": "Average spend per purchase (obs window)",
            "max_amount": "Max spend in observation window",
            "recency_days": "Days since last purchase at cutoff",
            "avg_gap_days": "Avg days between purchases (obs window)",
            "label_churn": "1 if NO purchase in label window; else 0"
        }
    }
    save_json(data_dict, OUT_DIR / "data_dictionary.json")
    meta = {
        "run_timestamp": datetime.now().isoformat(timespec="seconds"),
        "rng_seed": RNG_SEED,
        "params": {
            "cutoff": str(cutoff.date()),
            "obs_days": obs_days,
            "label_days": label_days
        },
        "rows": {
            "features": int(len(feats)),
            "sales_obs": int(len(sales_obs)),
            "daily_obs": int(len(daily_obs))
        }
    }
    save_json(meta, OUT_DIR / "run_metadata.json")

    qa = {
        "n_customers_total": int(customers_v["customer_id"].nunique()),
        "n_customers_with_obs_purchases": int(sales_obs["customer_id"].nunique()),
        "n_features_rows": int(len(feats)),
        "churn_rate": float(feats["label_churn"].mean()),
        "obs_window_start": str((cutoff - timedelta(days=obs_days)).date()),
        "cutoff": str(cutoff.date()),
        "label_window_end": str((cutoff + timedelta(days=label_days)).date())
    }
    save_json(qa, OUT_DIR / "qa_report.json")
    return qa

qa = dump_outputs(feats, sales_obs, daily_obs, default_cutoff, obs_days, label_days)
qa

#### Extract (II) and EDA checks

In [ ]:
import pandas as pd

# Load the saved features file
# df = pd.read_csv("output/features.csv")
# df = pd.read_parquet("output/features.parquet")
df = pd.read_json("output/features.json")

# Quick EDA to confirm shape and missingness
print(df.shape)
print(df.isnull().sum())
print(df.head(10))

#### Load (II)

In [ ]:
df.to_csv("data.csv", index=False)

#### Transform (II)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Load data
_df = pd.read_csv("data.csv")

# Separate features and target
X = _df.drop(columns=["label_churn", "customer_id"])
y = _df["label_churn"]

# Split BEFORE fitting anything (avoid leakage)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Infer column types from TRAIN ONLY
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

# Pipelines
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),  # dense
])

# Build ColumnTransformer (skip empty groups gracefully)
transformers = []
if numeric_cols:
    transformers.append(("num", numeric_pipeline, numeric_cols))
if categorical_cols:
    transformers.append(("cat", categorical_pipeline, categorical_cols))

preprocessor = ColumnTransformer(transformers)

# Fit on train, transform both
X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)

# Feature names
num_features = numeric_cols
cat_features = []
if categorical_cols:
    cat_features = preprocessor.named_transformers_["cat"]["encoder"]\
                               .get_feature_names_out(categorical_cols).tolist()
all_features = num_features + cat_features

# Back to DataFrames with aligned indices
X_train_df = pd.DataFrame(X_train_processed, columns=all_features, index=X_train.index).reset_index(drop=True)
X_valid_df = pd.DataFrame(X_valid_processed, columns=all_features, index=X_valid.index).reset_index(drop=True)

# Reattach target
train_final_df = pd.concat([X_train_df, y_train.reset_index(drop=True)], axis=1)
valid_final_df = pd.concat([X_valid_df, y_valid.reset_index(drop=True)], axis=1)

In [ ]:
# Save final datasets (Load - III)

# OR

# CAN BE USED FOR MODEL TRAINING AND VALIDATION DIRECTLY FROM HERE

#### Scaling to PySpark: lazy evaluation, shuffles, partitioning

Moving a data pipeline from single-node (pandas/SQL) to a distributed engine (PySpark) isn't a syntax port — it's rethinking transformation order around data-movement cost, a cost pandas never had to consider on a single machine.

**Not a 1:1 Port — Lazy Evaluation and Shuffle-Aware Ordering**

1. Joins, aggregations, and window functions get rewritten against Spark's lazy-evaluation DataFrame API rather than translated method-for-method from pandas.
2. Transformation order gets restructured specifically to minimize shuffles — expensive cross-node data movement — which pandas never has to reason about on one machine.

In [ ]:
# Illustrative -- not executed (pyspark not installed here)
# pandas version: joins/filters execute immediately, in-memory, single machine
# df_pandas.merge(other_df, on="user_id").groupby("day").agg({"amount": "sum"})

# PySpark equivalent: lazy, builds a query plan first, executes distributed on .collect()/.write()
# df_spark.join(other_df, on="user_id").groupBy("day").agg(F.sum("amount"))
# -- filtering BEFORE the join (predicate pushdown) shrinks what gets shuffled across the join,
#    which is the kind of reordering that has no pandas equivalent to even think about

**What a Shuffle Actually Is, and Why It Dominates Cost**

1. A shuffle happens when data needs to move across nodes to bring matching keys together — for a `groupBy` or `join` on a key that isn't already co-located.
2. It's expensive because it means network I/O and disk spill, not just CPU work — minimizing shuffles (not just minimizing operation count) is the real optimization target in distributed pipelines.
3. Filtering to the relevant rows *before* a join (predicate pushdown), instead of after, is what actually produces most of the real-world overhead reduction from a migration like this.

In [ ]:
# Genuinely computable: rough shuffle-volume estimate for a groupBy/join,
# to reason about WHY reordering operations to filter-before-join reduces cost

def estimate_shuffle_bytes(n_rows, avg_row_bytes, filter_selectivity=1.0):
    """Rough estimate of data volume moved across the network during a shuffle."""
    rows_after_filter = int(n_rows * filter_selectivity)
    return rows_after_filter * avg_row_bytes

n_rows = 50_000_000
avg_row_bytes = 120

no_pushdown = estimate_shuffle_bytes(n_rows, avg_row_bytes, filter_selectivity=1.0)          # join first, filter after
with_pushdown = estimate_shuffle_bytes(n_rows, avg_row_bytes, filter_selectivity=0.08)        # filter first (8% selectivity), then join

print(f"Shuffle volume without predicate pushdown: {no_pushdown / 1e9:.2f} GB")
print(f"Shuffle volume with predicate pushdown:    {with_pushdown / 1e9:.2f} GB")
print(f"Reduction: {(1 - with_pushdown / no_pushdown):.1%}")

**Why Distribute at All, Instead of Scaling One Machine Up**

1. Vertical scaling has a hard ceiling — there's a largest machine you can rent or own, and single-machine pandas is also single-threaded for most operations regardless of available RAM.
2. Horizontal scaling (a cluster) has no such ceiling and is more cost-elastic, since a batch job's cluster only needs to scale up for the duration of the run.
3. Honest caveat: horizontal scaling isn't free — it introduces the shuffle-cost problem above that a single machine never had. The real tradeoff isn't "horizontal is strictly better," it's that data volume crosses a point where vertical scaling's ceiling and cost curve lose to horizontal's coordination overhead.

**Partition Count and Cluster Sizing**

1. Target a partition size in the 100–200MB range — small enough for parallelism across executors, large enough that per-partition overhead doesn't dominate.
2. Too few partitions underuses the cluster; too many adds scheduling overhead per task.
3. This is a starting heuristic, not a fixed rule — the right answer also depends on cluster core count (partitions should be a multiple of total cores for even scheduling) and is usually tuned empirically from actual job execution time.

In [ ]:
def recommend_partition_count(total_data_gb, target_partition_mb=150):
    total_mb = total_data_gb * 1024
    return max(1, round(total_mb / target_partition_mb))

for size_gb in [5, 50, 500]:
    n_partitions = recommend_partition_count(size_gb)
    print(f"{size_gb} GB dataset -> ~{n_partitions} partitions "
          f"(~{size_gb*1024/n_partitions:.0f} MB each)")

**Measuring the Improvement Honestly**

1. Measure wall-clock time for the specific stage that changed (e.g. data-prep), pre- vs. post-migration, on comparable data volume.
2. Isolate that stage from downstream steps so the number reflects the pipeline change itself, not unrelated infrastructure improvements happening at the same time — a claimed improvement that coincides with an unrelated cluster upgrade isn't attributable to the migration alone.

#### Feature engineering at scale with PySpark

Behavior features (counts and sums over the last 7 or 30 days per customer or device) are the core of fraud and propensity models, and they are the expensive part of data preparation. The pattern is the same in pandas and Spark: group by the entity, order by time, aggregate over a trailing window. The pandas version below runs on a sample. At hundreds of millions of events the same logic moves to Spark window functions:

```python
from pyspark.sql import functions as F, Window
w = Window.partitionBy("user").orderBy(F.col("ts").cast("long")).rangeBetween(-7 * 86400, 0)   # trailing 7 days
feats = (events.select("user", "ts", "amount")                 # drop unused columns before the shuffle
               .withColumn("txn_7d", F.count("*").over(w))
               .withColumn("amount_7d", F.sum("amount").over(w)))
```

What keeps it fast: select only needed columns early, partition by the entity once and reuse that partitioning, use built-in functions instead of Python UDFs, broadcast small lookup tables in joins, watch for skewed keys (one very active account), cache only data that is reused, and write the output as partitioned Parquet. Measure with the Spark UI (shuffle size and stage time) before and after, so a claim like "60% less preparation time" is a measured one.

In [1]:
import numpy as np, pandas as pd
rng = np.random.default_rng(0)
n = 20000
ev = pd.DataFrame({"user": rng.integers(0, 200, n),
                   "ts": pd.Timestamp("2025-01-01") + pd.to_timedelta(rng.integers(0, 60 * 24 * 30, n), unit="m"),
                   "amount": rng.gamma(2, 40, n)}).sort_values(["user", "ts"])
feat = (ev.set_index("ts").groupby("user")["amount"].rolling("7D").agg(["count", "sum"])
          .rename(columns={"count": "txn_7d", "sum": "amount_7d"}).reset_index())
print(feat.groupby("user").tail(1).head(5).round(1).to_string(index=False))

 user                  ts  txn_7d  amount_7d
    0 2025-01-30 14:20:00    17.0     1397.0
    1 2025-01-30 21:27:00    24.0     1474.8
    2 2025-01-30 19:27:00    20.0     1870.5
    3 2025-01-30 18:08:00    23.0     2153.6
    4 2025-01-30 18:04:00    30.0     2374.1


### Model Serving and Deployment

The Data Pipelines section above covers extract/transform/load and training. This picks up after that: you have a trained model artifact, how does it actually serve predictions to something else, an app, a batch job, another service. Covers batch vs. real-time inference, serialization, wrapping a model as an API, containerizing it, and what changes once it's live (versioning, monitoring).

#### Batch vs. real-time inference

Batch: run the model over a large set of rows on a schedule (nightly, hourly), write results somewhere (a table, a file), nothing waits on it in real time. Real-time (online): a request comes in, the model scores it, a response goes back, all within the caller's own request, latency is now a hard constraint, not just a nice-to-have.

Worked comparison, fraud scoring: batch fits "re-score every account overnight, flag anything above a threshold for tomorrow's review queue", cheap, simple, but a fraudulent transaction at 2pm doesn't get caught until the next batch run. Real-time fits "score this transaction before it's authorized", catches it immediately, but now the model has to respond in milliseconds, under load, with an uptime guarantee, a much higher engineering bar for the exact same model artifact. Same model, completely different deployment shape, chosen by the LATENCY requirement of the decision it feeds, not by anything about the model itself.

#### Serialization: saving a trained model as a portable artifact

Different formats for different frameworks, all solving the same problem, getting a trained model's parameters out of the training process and into something a separate serving process can load without retraining.

- `pickle`/`joblib`: standard for sklearn/XGBoost, `joblib` preferred for anything with large numpy arrays (more efficient than raw pickle for that case). Tied to the exact Python/library versions used to train, a real operational risk, loading a joblib file with a different sklearn version than it was saved with can silently break or refuse to load.
- PyTorch: `state_dict` (the model's learned weights only, not the class definition), covered in `01-nn-training.ipynb`'s "Saving and loading models" section, requires the model class definition to exist wherever it's loaded.
- ONNX: a framework-agnostic format, export a PyTorch or sklearn model to ONNX once, then run it in any ONNX-compatible runtime (C++, Java, a mobile device) without needing Python or the original framework installed at serving time, trades some flexibility for portability and often faster inference (the ONNX runtime is optimized specifically for inference, not training).

In [ ]:
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=200, n_features=4, random_state=42)
model = RandomForestClassifier(random_state=42).fit(X, y)

joblib.dump(model, "fraud_model.joblib")
loaded_model = joblib.load("fraud_model.joblib")

print("original vs loaded predictions match:", (model.predict(X) == loaded_model.predict(X)).all())

#### Wrapping a model as a real-time API (FastAPI)

FastAPI pairs naturally with `00-python-tooling/04-pydantic-basics.ipynb`, the request/response schemas ARE pydantic models, validation (bad request shape, wrong types, out-of-range values) happens automatically before your prediction code ever runs, the same validation mechanism from that notebook, now sitting at the boundary of a live service instead of validating a dict in a script.

Load the model ONCE at startup (module-level, not inside the endpoint function), loading a joblib/ONNX file from disk on every request would add real latency and is pure waste, the model doesn't change between requests.

In [ ]:
# app.py — illustrative, not executed here (fastapi/uvicorn not installed in this env)
# run with: uvicorn app:app --reload

# from fastapi import FastAPI
# from pydantic import BaseModel, Field
# import joblib
# import numpy as np
#
# app = FastAPI()
# model = joblib.load("fraud_model.joblib")  # loaded once at startup, not per-request
#
# class TransactionRequest(BaseModel):
#     amount: float = Field(..., ge=0)
#     narrative_length: int = Field(..., ge=0)
#     mentions_irs: bool
#     hour_of_day: int = Field(..., ge=0, le=23)
#
# class FraudScoreResponse(BaseModel):
#     fraud_probability: float
#     flagged: bool
#
# @app.post("/predict", response_model=FraudScoreResponse)
# def predict(txn: TransactionRequest):
#     features = np.array([[txn.amount, txn.narrative_length, int(txn.mentions_irs), txn.hour_of_day]])
#     prob = model.predict_proba(features)[0, 1]
#     return FraudScoreResponse(fraud_probability=float(prob), flagged=prob > 0.5)

print("see commented app.py above — a malformed request (e.g. hour_of_day=99) gets rejected")
print("by pydantic validation automatically, the endpoint function never even runs")

#### Containerizing with Docker

The problem it solves: "works on my machine" — the serving environment needs the exact same Python version, library versions, and OS-level dependencies the model was trained/tested against, a container packages all of that into one portable image instead of hoping the deployment target happens to match.

Worked minimal Dockerfile for the FastAPI app above:
```dockerfile
FROM python:3.12-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py fraud_model.joblib ./

EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```
Line by line: `FROM` picks a minimal base image (smaller image, smaller attack surface, faster pulls than a full OS image). `WORKDIR` sets the working directory inside the container. Dependencies are installed in their OWN layer, before copying application code, so Docker's layer cache can skip the (slow) `pip install` step on rebuilds where only `app.py` changed, not `requirements.txt`, a real build-speed detail, not just style. `EXPOSE` documents the port (doesn't actually publish it, that's done with `docker run -p`). `CMD` is what runs when the container starts.

#### Model registry and versioning

Once a model is deployed, "which exact model artifact is live" becomes a real question, tied to the Drift Monitoring section's retraining triggers, when drift fires and a new model gets trained, something needs to track: which training data version produced it, what its validation metrics were, whether it's currently in production or staging, and how to roll back to the previous version if the new one underperforms live.

A model registry (MLflow's Model Registry is the common open-source one) stores exactly this: each trained model gets a version number, metadata (metrics, training data snapshot, timestamp), and a stage label (`staging`/`production`/`archived`). Promoting a model from staging to production becomes a metadata change, not a redeploy, and rollback is just re-pointing production at the previous version number, the artifact never disappeared.

#### Scaling and monitoring a live model

Scaling: run multiple copies of the serving process behind a load balancer (horizontal scaling, same idea as scaling a normal web service), stateless by design, each request is independent, the model itself doesn't need to know about other replicas. For GPU-bound models specifically, request batching (grouping several incoming requests into one forward pass) trades a small amount of added latency per request for much higher total throughput, since a GPU forward pass over a batch of 8 isn't much slower than over 1.

Monitoring: two different things people mean by "monitoring" a model. Infra-level (latency, error rate, uptime, ordinary service monitoring). Model-level (is it still ACCURATE, the Drift Monitoring section's PSI/KS-test territory, checking whether the live input distribution or the relationship between features and outcome has shifted since training). A model can be perfectly healthy on every infra metric, 100% uptime, fast responses, while being silently wrong on every prediction, infra monitoring alone can't see that, only the drift/label-quality checks can.

Canary/shadow deployment: roll a new model out to a small slice of traffic (canary) or run it alongside the current production model without actually using its predictions yet (shadow), comparing its outputs to the incumbent's before trusting it with 100% of traffic, the safe way to validate "does this actually work in production" without betting the whole system on it.

#### Batch vs. real-time, summarized

| | Batch | Real-time |
|---|---|---|
| Latency requirement | none, runs on a schedule | milliseconds to seconds |
| Infra complexity | low, a scheduled job | higher, needs uptime/scaling/load balancing |
| Freshness | stale between runs (hours) | immediate |
| Typical fit | overnight fraud re-scoring, reporting | transaction authorization, live recommendations |
| Failure mode | a late/failed job delays everyone | a slow/down service blocks the caller directly |

### Drift Monitoring and Retraining

A model's performance at deployment time is not permanent, the world the model was trained on keeps changing. Covers the types of drift, how to detect them (PSI, KS-test), what to monitor in production, and how to decide when to retrain.

#### Types of drift

Data drift (covariate shift): the distribution of INPUT features changes, but the true relationship between features and target stays the same. Example: a fraud model trained when `amount_lost` averaged $2,000 now sees transactions averaging $8,000, because inflation or a new user segment, the fraud mechanics themselves have not changed, but the model's training distribution no longer matches production.

Concept drift: the relationship between features and target itself changes, the same input now means something different. Example: `payment_method_requested=gift_card` used to be a strong fraud signal, but if legitimate businesses started commonly requesting gift card payments for unrelated reasons, that same feature value now means something different than it did at training time.

Label drift: the distribution of the TARGET itself shifts, independent of feature drift, e.g. a genuinely new fraud typology emerges that did not exist in training data at all (directly relevant to the fraud-theme-detection project, trained on a fixed list of 10 known typologies, a new scam pattern would not even have a label to be assigned to).

#### PSI (Population Stability Index), worked by hand

Formula: PSI = sum over bins of (actual% - expected%) * ln(actual% / expected%), where "expected" is the reference (training) distribution and "actual" is the current (production) distribution.

Worked example, feature = amount_lost, binned into 3 buckets:
```
                low    medium   high
reference:      50%    30%      20%
current:        30%    30%      40%

PSI = (0.30-0.50)*ln(0.30/0.50) + (0.30-0.30)*ln(0.30/0.30) + (0.40-0.20)*ln(0.40/0.20)
    = (-0.20)*ln(0.6) + 0*ln(1) + (0.20)*ln(2.0)
    = (-0.20)*(-0.511) + 0 + (0.20)*(0.693)
    = 0.102 + 0 + 0.139
    = 0.241
```
Standard interpretation thresholds: PSI < 0.1 = no significant shift, 0.1-0.25 = moderate shift, worth investigating, PSI > 0.25 = significant shift, retraining likely needed. 0.241 lands in "moderate, investigate" territory, right at the edge of the retrain threshold, exactly the kind of borderline case a monitoring dashboard should flag for a human to look at rather than silently ignore or silently auto-retrain on.

In [ ]:
import numpy as np

reference = np.array([0.50, 0.30, 0.20])
current = np.array([0.30, 0.30, 0.40])

psi_per_bin = (current - reference) * np.log(current / reference)
psi = np.sum(psi_per_bin)

print("PSI per bin:", psi_per_bin.round(3))
print("total PSI:", psi.round(3))
print("interpretation:", "stable" if psi < 0.1 else ("investigate" if psi < 0.25 else "retrain"))

#### KS-test (Kolmogorov-Smirnov), worked by hand

The KS statistic is the maximum vertical distance between two empirical CDFs (cumulative distribution functions), a distribution-free way to compare a reference sample and a current sample without binning.

Worked example, tiny samples: reference=[1,2,3,4,5], current=[3,4,5,6,7] (a clean shift upward).
```
value:        1    2    3    4    5    6    7
F_reference:  0.2  0.4  0.6  0.8  1.0  1.0  1.0
F_current:    0.0  0.0  0.2  0.4  0.6  0.8  1.0
difference:   0.2  0.4  0.4  0.4  0.4  0.2  0.0
```
Maximum difference (the KS statistic, D) = 0.4, occurring at several points (values 2 through 5). A larger D means the two distributions differ more; a p-value (from the KS distribution given both sample sizes) then says whether that D is large enough to be statistically significant rather than just sampling noise, not covered in detail here, the intuition (max gap between two CDFs) is the part worth internalizing.

In [ ]:
from scipy import stats

reference_sample = [1, 2, 3, 4, 5]
current_sample = [3, 4, 5, 6, 7]

ks_stat, p_value = stats.ks_2samp(reference_sample, current_sample)
print("KS statistic:", ks_stat)
print("p-value:", p_value)

#### What to actually monitor in production

- Feature distributions: PSI/KS-test per feature, catches data drift before it shows up as a performance drop.
- Prediction distributions: is the model's OUTPUT distribution shifting (e.g. suddenly predicting a much higher rate of one class), even without knowing the true labels yet, a real early-warning signal.
- Performance metrics, once labels arrive: precision/recall/F1 (see `06-metrics-selection-tuning.ipynb`) computed on a rolling window of recent labeled data, the direct measure, but delayed by however long it takes labels to actually arrive (in fraud, labels come from chargebacks/confirmed reports, which can lag the original transaction by weeks).
- Calibration drift (see `06-metrics-selection-tuning.ipynb`'s calibration section): even if ranking (AUC) stays fine, the model's probabilities can become miscalibrated over time as the underlying rates shift, silently breaking any downstream threshold-based decision.

#### Retraining triggers

Threshold-based: retrain when PSI/KS crosses a set level on a monitored feature, or when a performance metric drops below a set floor. Reactive but precise, ties retraining directly to evidence of actual drift.

Scheduled: retrain on a fixed calendar (weekly, monthly) regardless of measured drift. Simple, predictable, but wastes compute retraining when nothing has changed, and can still miss a sudden drift event that happens mid-cycle.

Performance-based: retrain when live performance metrics (once labels catch up) fall below a floor, the most direct trigger, but the most delayed, by definition it only fires after the model has already been degraded in production for however long labels take to arrive.

Practical pattern: combine them, PSI/KS as an early warning (fast, no label lag) that triggers investigation, performance metrics as the final confirmation before committing to a full retrain, since drift in a feature does not always translate into an actual performance drop, some drifted features may not matter much to the model's decision boundary.

#### Automated retraining triggers

Three triggers are common: a schedule (every 90 days), data drift (PSI on the score or key features above 0.25), and performance decay measured on matured labels (AUC or precision below a floor). Drift is available immediately but only suggests a problem, while performance is the real signal but arrives late because labels do. Use both.

A trigger should start a retraining job, not push a new model straight to production. The new model is a challenger: evaluate it against the current champion on a holdout or in shadow mode, require an approval step where governance demands one, and keep the previous version registered so that rollback is one alias change.

Twelve simulated weeks of live data where the population shifts from week 4:

In [2]:
import numpy as np
rng = np.random.default_rng(0)

def psi(expected, actual, bins=10):
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1)); edges[0], edges[-1] = -np.inf, np.inf
    e = np.clip(np.histogram(expected, edges)[0] / len(expected), 1e-6, None)
    a = np.clip(np.histogram(actual, edges)[0] / len(actual), 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))

def retrain_reasons(psi_value, auc, days_since_train, auc_floor=0.80):
    reasons = []
    if psi_value > 0.25: reasons.append("score drift (PSI above 0.25)")
    if auc is not None and auc < auc_floor: reasons.append(f"AUC {auc:.2f} below {auc_floor}")
    if days_since_train >= 90: reasons.append("scheduled retrain")
    return reasons

baseline = rng.normal(0, 1, 20000)
for week in range(0, 12, 2):
    live = rng.normal(0.0 if week < 4 else 0.25 * (week - 3), 1, 5000)
    auc = None if week < 4 else 0.86 - 0.015 * (week - 3)              # AUC is only known once labels have matured
    p = psi(baseline, live)
    print(f"week {week:2d} | PSI {p:.3f} | AUC {'n/a' if auc is None else f'{auc:.2f}'} | trigger: {retrain_reasons(p, auc, week * 7) or 'none'}")

week  0 | PSI 0.003 | AUC n/a | trigger: none
week  2 | PSI 0.001 | AUC n/a | trigger: none
week  4 | PSI 0.063 | AUC 0.84 | trigger: none
week  6 | PSI 0.483 | AUC 0.81 | trigger: ['score drift (PSI above 0.25)']
week  8 | PSI 1.387 | AUC 0.79 | trigger: ['score drift (PSI above 0.25)', 'AUC 0.79 below 0.8']
week 10 | PSI 2.649 | AUC 0.76 | trigger: ['score drift (PSI above 0.25)', 'AUC 0.76 below 0.8']


### MLOps Tooling and Automation

#### Experiment tracking and model registry (MLflow)

MLflow Tracking records, for every run, the parameters, metrics, code version and artifacts, so any result can be compared and reproduced. The Model Registry gives each registered model a name and numbered versions, and aliases such as `champion` and `challenger` are named pointers to a version. Aliases replaced the older registry stages (deprecated in MLflow 2.9). Serving code loads `models:/fraud-model@champion`, so promoting a new version means moving the alias, not editing the application.

In a regulated setting the run history plus the registry is the audit trail: which data, code and parameters produced the model that was approved, who approved it, and which version is live. Minimal flow: log runs, register the best one, point `champion` at it, load by alias.

In [ ]:
import tempfile, mlflow
from mlflow.tracking import MlflowClient
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

mlflow.set_tracking_uri(f"sqlite:///{tempfile.mkdtemp()}/mlflow.db")      # the registry needs a database-backed store
mlflow.set_experiment("fraud-demo")
X, y = make_classification(5000, 20, weights=[0.97], random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

for depth in (2, 4):
    with mlflow.start_run(run_name=f"gbm-depth-{depth}"):
        model = GradientBoostingClassifier(max_depth=depth, random_state=0).fit(Xtr, ytr)
        mlflow.log_params({"max_depth": depth})
        mlflow.log_metric("auc", roc_auc_score(yte, model.predict_proba(Xte)[:, 1]))
        mlflow.sklearn.log_model(model, name="model", registered_model_name="fraud-model")

client = MlflowClient()
versions = client.search_model_versions("name='fraud-model'")
best = max(versions, key=lambda v: client.get_run(v.run_id).data.metrics["auc"])
client.set_registered_model_alias("fraud-model", "champion", best.version)   # promotion = move the alias
champion = mlflow.sklearn.load_model("models:/fraud-model@champion")
print("champion is version", best.version, "| predictions:", champion.predict(Xte[:3]))

#### CI/CD for ML (GitHub Actions)

A model pipeline needs the usual software checks plus checks on data and model quality. A GitHub Actions workflow runs on a pull request (and on a schedule) and fails the build when a gate is not met:

```yaml
name: ml-ci
on:
  pull_request:
  schedule:
    - cron: "0 2 * * 1"                 # weekly check that retraining still passes the gates
jobs:
  test-and-gate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.12" }
      - run: pip install -r requirements.txt
      - run: ruff check . && pytest -q                         # code, feature logic and schema/data validation tests
      - run: python train.py --sample 0.05                     # smoke-train on a small sample
      - run: python evaluate.py --min-auc 0.80 --max-psi 0.25  # quality gate: non-zero exit fails the job
      - run: docker build -t fraud-api:${{ github.sha }} .     # same image from test to production
```

What differs from ordinary CI: tests on data (schema, ranges, leakage checks), metric gates against the current champion, reproducible environments (pinned dependencies, container image), promotion through the model registry instead of copying files, and deployment through shadow or canary stages with an approval step (a protected environment in GitHub) where governance requires sign-off. Secrets stay in the repository's secret store, never in code.

#### Dataiku: flow, scenarios and automation

Dataiku is a platform that puts the pipeline, the model and the operations in one place. Its main ideas map onto things covered above:

- **Flow.** A graph of datasets and recipes (visual or code), like a pipeline DAG.
- **Scenarios.** Automation: a set of steps run on a trigger (a schedule, a dataset change, or a metric leaving its allowed range). A scenario can rebuild the flow daily, or retrain a model when a drift or performance metric crosses a threshold.
- **Nodes.** A design node for building, an automation node for running production flows and scenarios, and an API node for real-time scoring over REST. Separating them keeps experimentation away from production.
- **Saved models and training recipes.** A deployed model comes with a recipe that retrains it on new data with the same settings, with versions kept for rollback and comparison.
- **Monitoring and governance.** Drift and performance are tracked over time, and project sign-off workflows support model review.

The practical point: the concepts (retraining triggers, champion and challenger, separate dev and prod, audit trail) are the same as in an MLflow, Airflow and GitHub Actions stack. What changes is how much the platform provides and how much you wire yourself.

#### Cloud basics for ML on AWS

| Need | AWS building block | Notes |
|---|---|---|
| Data and artifacts | S3 | versioning, lifecycle rules, encryption with KMS |
| Access control | IAM roles and policies | least privilege, separate roles for training and serving |
| Training | SageMaker training jobs, or EC2 / EKS | spot instances cut cost for interruptible training |
| Real-time scoring | SageMaker endpoint, or a container on ECS / EKS / Lambda | autoscaling, latency budget, canary deployments |
| Batch scoring | SageMaker batch transform, or scheduled jobs | cheapest when results are not needed instantly |
| Pipelines and registry | SageMaker Pipelines and Model Registry | approval status before deployment |
| Monitoring | CloudWatch, SageMaker Model Monitor | logs, latency and error metrics, data and model drift |
| Network and security | VPC, private endpoints, KMS | no public path to regulated data |

Decision rules: real-time endpoint for decisions on request, batch for scheduled lists, serverless or containers for small and spiky workloads. In a bank, security and data residency usually decide the design before cost does: encrypt everything, keep data in approved regions, log access, and keep training and serving roles separate.

#### Dashboards and BI for model monitoring (Power BI)

A monitoring dashboard for stakeholders answers three questions: is the system healthy, is it still accurate, and is anything changing. Typical tiles: volume and latency, alert or flag rate, precision at review capacity and review backlog (once labels mature), PSI for the score and key features over time, and the share of decisions by action (allow, review, block).

In Power BI the data goes into a semantic model, measures are written in DAX (for example a safe ratio with `DIVIDE(confirmed, reviewed)`), and refresh is scheduled (import mode) or live (DirectQuery) depending on data volume and freshness needs. Row-level security restricts what each audience sees. Keep out vanity metrics (raw alert counts without precision) and always show the time window and the label maturity, since the latest weeks are incomplete.